<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/05_Retrieval_mechanisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Langchain/

Mounted at /content/drive
/content/drive/MyDrive/Langchain


In [ ]:
# #Python 3.12.13
# !pip install -U langchain==1.3.11
# !pip install requests==2.34.2
# !pip install -U langchain_community==0.4.2
# !pip install -U python-dotenv==1.2.2
# !pip install -U langchain-huggingface==1.2.2
# !pip install faiss-cpu==1.14.3
# !pip install langchain-groq==1.1.3
# !pip install chromadb==1.5.9
# !pip install -U flashrank==0.2.10
# !pip install langchain-classic==1.0.8
!pip install rank_bm25==0.2.2

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS ,Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

/tmp/ipykernel_1275/42218320.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


True

In [ ]:
docs = WebBaseLoader(["https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag" ,
                      "https://aws.amazon.com/what-is/retrieval-augmented-generation/"]).load()

Re-ranking [with Rag Fusion]

In [ ]:
# spliting
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
# embedding modle
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)

#  create vDB
vector_db = Chroma.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# define the ranking functoin , retunes each doc and the rank
def rank_retrived_docs(documents_list= list[list]  , k =60) :
  fused_scores = dict()

  for docs in documents_list :
    for rank , doc in enumerate(docs) :
      key =(doc.page_content ,
            frozenset(doc.metadata.items()))

      if key not in fused_scores:
        fused_scores[key] = {
            "document": doc,
            "score": 0.0,
        }

      fused_scores[key]["score"] += 1 / (rank + k)


  ranked_resualts = sorted(
      ( (item['document'] , item['score'])
      for item in fused_scores.values()) ,
      key= lambda x : x[1] ,
      reverse=True)
  return ranked_resualts

In [ ]:
# genrate queries
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to this question: {question} \n
only Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)
generate_queries = (
    prompt_rag_fusion
    | ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [ ]:
generate_queries.invoke("what is Rag?")

['1. What is the definition of Rag music',
 '2. Rag as a type of fabric or textile',
 '3. Rag in the context of Indian classical music',
 '4. What is a Rag doll or its significance in toys']

In [ ]:
# path to llm wih it's order , cause llm can't lose context  , making high proirity doc in first

get_ranked_docs = (
    generate_queries
    |retriever.map()
    |rank_retrived_docs
)


In [ ]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

rag_fusion_chain = (
    {
        "context" : get_ranked_docs ,
        "question" : RunnablePassthrough()
    }
    | prompt
    | ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
    | StrOutputParser()
)

rag_fusion_chain.invoke("what is Rag?")

"Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language model, so it references an authoritative knowledge base outside of its training data sources before generating a response. It extends the already powerful capabilities of Large Language Models (LLMs) to specific domains or an organization's internal knowledge base, all without the need to retrain the model. This approach allows developers to provide the latest research, statistics, or news to the generative models, enabling them to present accurate information with source attribution and increasing user trust and confidence in the generative AI solution."

Re-rank [with Flash Re-Rank]

In [ ]:
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever
from flashrank import Ranker , RerankRequest
FlashrankRerank.model_rebuild()
compressor = FlashrankRerank( model="ms-marco-MiniLM-L-12-v2")

vector_db = Chroma.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever()

compression_retriever = ContextualCompressionRetriever(base_compressor= compressor , base_retriever=retriever)

compression_retriever.invoke("what is Rag?")

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 72.1MiB/s]


[Document(metadata={'id': 0, 'relevance_score': np.float32(0.99769884), 'title': 'What is RAG? - Retrieval-Augmented Generation AI Explained - AWS', 'source': 'https://aws.amazon.com/what-is/retrieval-augmented-generation/', 'language': 'en-US', 'description': 'What is Retrieval-Augmented Generation (RAG), how and why businesses use RAG AI, and how to use RAG with AWS.'}, page_content='What is RAG? - Retrieval-Augmented Generation AI Explained - AWS\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to main content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n     \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nFilter: All\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nEnglish\n\n\n\n\n\nContact us\nAWS Marketplace\n\nSupport  \n\n\n\n\n\n\nMy account 

# Ensemble Retrieval

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever



bm_retriever = BM25Retriever.from_documents(chunks)
denese_retriever = vector_db.as_retriever()

ensemble_retriever =  EnsembleRetriever(retrievers = [bm_retriever,denese_retriever] , weights=[0.5,0.5])
ensemble_retriever.invoke("what is Rag?")

[Document(metadata={'language': 'en-US', 'title': 'What is RAG? - Retrieval-Augmented Generation AI Explained - AWS', 'source': 'https://aws.amazon.com/what-is/retrieval-augmented-generation/', 'description': 'What is Retrieval-Augmented Generation (RAG), how and why businesses use RAG AI, and how to use RAG with AWS.'}, page_content='What is RAG? - Retrieval-Augmented Generation AI Explained - AWS\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to main content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n     \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nFilter: All\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nEnglish\n\n\n\n\n\nContact us\nAWS Marketplace\n\nSupport  \n\n\n\n\n\n\nMy account  \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n     

# long context recorder

In [ ]:
from langchain_community.document_transformers import LongContextReorder

In [ ]:
long_context_reorder = LongContextReorder()
long_context_chain =  (
    retriever |
    long_context_reorder.transform_documents

)


long_context_chain.invoke("what is rag")

In [ ]:
retriever.invoke("what is rag")

# SELF Query , meta-data retriever

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]
vectorstore = Chroma.from_documents(docs, embedding_model)

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

In [ ]:
# to solve some packages versions problems , making empty fake placeholers classes
import langchain_community.vectorstores as _lcv

_missing = ["DatabricksVectorSearch", "DeepLake", "Milvus", "Neo4jVector",
            "Qdrant", "Weaviate", "MongoDBAtlasVectorSearch", "Pinecone"]

for _name in _missing:
    if not hasattr(_lcv, _name):
        setattr(_lcv, _name, type(_name, (), {}))  # inert placeholder class

from langchain_classic.retrievers import SelfQueryRetriever
from langchain_classic.retrievers.self_query.base import AttributeInfo

In [ ]:
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_classic.retrievers.self_query.base import AttributeInfo
llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
document_content_dec = "summary of a movie"
retreiver = SelfQueryRetriever.from_llm(
    llm = llm ,
    vectorstore = vectorstore ,
    document_contents=document_content_dec ,
    metadata_field_info=  metadata_field_info
)

In [ ]:
retreiver.invoke("I want to watch a movie made by Greta Gerwig")

[Document(metadata={'year': 2019, 'director': 'Greta Gerwig', 'rating': 8.3}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them')]